# ROGII — Wellbore Geology Prediction
## Notebook 2: Feature Engineering + Modeling + Submission

This notebook covers:
1. Feature engineering (raw, rolling, lag, DWT, trajectory)
2. 5-fold cross-validation split by well
3. LightGBM baseline → XGBoost → CatBoost → ensemble
4. Generating the Kaggle submission

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import QuantileTransformer

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import pywt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
sns.set_palette('husl')

DATA_DIR   = Path('../data')
SUBMIT_DIR = Path('../submissions')
SUBMIT_DIR.mkdir(exist_ok=True)

SEED = 42
N_FOLDS = 5

## 1. Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

# Column name constants — adjust these if the actual column names differ
WELL_COL   = 'well_id'
TARGET_COL = 'TVT'
DEPTH_COL  = 'DEPT'
GR_COL     = 'GR'
X_COL, Y_COL, Z_COL = 'X', 'Y', 'Z'

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train columns: {list(train.columns)}')

## 2. Feature Engineering

In [ ]:
def compute_tortuosity(df, x_col, y_col, z_col, depth_col, window=5):
    """
    3D wellbore tortuosity: ratio of actual path length to straight-line distance
    within a rolling window. High tortuosity = highly curved section.
    """
    dx = df[x_col].diff()
    dy = df[y_col].diff()
    dz = df[z_col].diff()
    
    ds = np.sqrt(dx**2 + dy**2 + dz**2).fillna(0)
    
    # Arc length over window
    arc = ds.rolling(window=window, min_periods=1).sum()
    
    # Straight-line distance: from position (t-window) to (t)
    x_shift = df[x_col] - df[x_col].shift(window)
    y_shift = df[y_col] - df[y_col].shift(window)
    z_shift = df[z_col] - df[z_col].shift(window)
    chord = np.sqrt(x_shift**2 + y_shift**2 + z_shift**2).fillna(arc)
    
    tortuosity = (arc / chord.replace(0, np.nan)).fillna(1.0)
    return tortuosity


def dwt_features(series, wavelet='db4', level=3):
    """
    Discrete Wavelet Transform on a 1D signal.
    Returns approximation and detail coefficients resampled to original length.
    """
    coeffs = pywt.wavedec(series.values, wavelet, level=level)
    results = {}
    for i, c in enumerate(coeffs):
        # Upsample coefficient array back to original length
        upsampled = np.interp(
            np.linspace(0, len(c) - 1, len(series)),
            np.arange(len(c)),
            c
        )
        label = 'approx' if i == 0 else f'detail_{i}'
        results[f'dwt_{label}'] = upsampled
    return pd.DataFrame(results, index=series.index)


def normalize_gr_per_well(df, gr_col, well_col):
    """Subtract per-well mean and divide by per-well std."""
    stats = df.groupby(well_col)[gr_col].agg(['mean', 'std']).rename(
        columns={'mean': 'gr_wmean', 'std': 'gr_wstd'})
    df = df.merge(stats, on=well_col, how='left')
    df['GR_norm'] = (df[gr_col] - df['gr_wmean']) / (df['gr_wstd'] + 1e-6)
    return df.drop(columns=['gr_wmean', 'gr_wstd'])


def engineer_features(df, well_col, depth_col, gr_col, x_col, y_col, z_col):
    """
    Full feature engineering pipeline.
    Processes each well independently to avoid leakage across wells.
    """
    df = df.copy()
    df = normalize_gr_per_well(df, gr_col, well_col)
    
    all_parts = []
    
    for well_id, gdf in df.groupby(well_col):
        gdf = gdf.sort_values(depth_col).copy()
        gr = gdf['GR_norm']  # use normalized GR for feature engineering
        
        # ── Rolling statistics ──────────────────────────────────────────
        for w in [3, 5, 10, 20, 50]:
            gdf[f'gr_roll_mean_{w}'] = gr.rolling(w, min_periods=1).mean()
            gdf[f'gr_roll_std_{w}']  = gr.rolling(w, min_periods=1).std().fillna(0)
            gdf[f'gr_roll_min_{w}']  = gr.rolling(w, min_periods=1).min()
            gdf[f'gr_roll_max_{w}']  = gr.rolling(w, min_periods=1).max()
        
        # ── Lag features ────────────────────────────────────────────────
        for lag in [1, 2, 3, 5, 10, 15, 20]:
            gdf[f'gr_lag_{lag}']  = gr.shift(lag)
            gdf[f'gr_lead_{lag}'] = gr.shift(-lag)  # future GR — available at submission
        
        # ── Difference (derivative) ─────────────────────────────────────
        gdf['gr_diff1'] = gr.diff(1).fillna(0)
        gdf['gr_diff2'] = gr.diff(2).fillna(0)
        gdf['gr_abs_diff1'] = gdf['gr_diff1'].abs()
        
        # ── DWT features ────────────────────────────────────────────────
        try:
            dwt = dwt_features(gr, wavelet='db4', level=3)
            for col in dwt.columns:
                gdf[col] = dwt[col].values
        except Exception:
            pass
        
        # ── 3D trajectory features ──────────────────────────────────────
        if all(c in gdf.columns for c in [x_col, y_col, z_col]):
            for w in [5, 10, 20]:
                gdf[f'tortuosity_{w}'] = compute_tortuosity(
                    gdf, x_col, y_col, z_col, depth_col, window=w)
            
            # Rate of change of trajectory
            gdf['dx'] = gdf[x_col].diff().fillna(0)
            gdf['dy'] = gdf[y_col].diff().fillna(0)
            gdf['dz'] = gdf[z_col].diff().fillna(0)
            gdf['ds'] = np.sqrt(gdf['dx']**2 + gdf['dy']**2 + gdf['dz']**2)
            
            # Dip angle (angle from horizontal)
            gdf['dip'] = np.degrees(np.arctan2(gdf['dz'].abs(), 
                                                np.sqrt(gdf['dx']**2 + gdf['dy']**2) + 1e-6))
        
        # ── Depth features ──────────────────────────────────────────────
        gdf['depth_norm'] = (gdf[depth_col] - gdf[depth_col].min()) / \
                            (gdf[depth_col].max() - gdf[depth_col].min() + 1e-6)
        
        # ── Position within well ────────────────────────────────────────
        gdf['well_pos'] = np.arange(len(gdf)) / len(gdf)
        
        all_parts.append(gdf)
    
    result = pd.concat(all_parts, ignore_index=True)
    return result


print('Engineering features for train set...')
train_fe = engineer_features(train, WELL_COL, DEPTH_COL, GR_COL, X_COL, Y_COL, Z_COL)
print('Engineering features for test set...')
test_fe  = engineer_features(test,  WELL_COL, DEPTH_COL, GR_COL, X_COL, Y_COL, Z_COL)

print(f'Train after FE: {train_fe.shape}')
print(f'Test  after FE: {test_fe.shape}')

## 3. Prepare Feature Matrix

In [ ]:
# Drop non-feature columns
drop_cols = [TARGET_COL, WELL_COL, 'row_id']  # add any ID columns here
feature_cols = [c for c in train_fe.columns
                if c not in drop_cols and train_fe[c].dtype != object]

X_train = train_fe[feature_cols].astype(np.float32)
y_train = train_fe[TARGET_COL].astype(np.float32)
groups  = train_fe[WELL_COL]

X_test = test_fe[feature_cols].astype(np.float32)

# Fill any remaining NaN with column medians
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test  = X_test.fillna(medians)

print(f'Feature matrix: {X_train.shape}  |  Target: {y_train.shape}')
print(f'Features: {feature_cols[:10]}...')

## 4. Cross-Validation Setup

> **Critical**: always group-split by well to avoid data leakage. A random split would mix rows from the same well across train/val, making validation scores optimistic and useless.

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)

splits = list(gkf.split(X_train, y_train, groups=groups))

print(f'{N_FOLDS}-fold GroupKFold by well:')
for fold_i, (tr_idx, val_idx) in enumerate(splits):
    tr_wells  = groups.iloc[tr_idx].nunique()
    val_wells = groups.iloc[val_idx].nunique()
    print(f'  Fold {fold_i}: train {len(tr_idx):>6} rows ({tr_wells} wells)  |  '
          f'val {len(val_idx):>6} rows ({val_wells} wells)')

## 5. Model A: LightGBM

In [ ]:
lgb_params = {
    'objective':        'regression',
    'metric':           'rmse',
    'learning_rate':    0.03,
    'num_leaves':       127,
    'max_depth':        -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     1,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'n_jobs':           -1,
    'verbosity':        -1,
    'seed':             SEED,
}

lgb_oof_preds  = np.zeros(len(X_train))
lgb_test_preds = np.zeros(len(X_test))
lgb_fold_rmse  = []

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )
    
    val_pred = model.predict(X_val)
    lgb_oof_preds[val_idx] = val_pred
    lgb_test_preds += model.predict(X_test) / N_FOLDS
    
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    lgb_fold_rmse.append(rmse)
    print(f'  Fold {fold_i} LGB RMSE: {rmse:.4f}')

lgb_oof_rmse = np.sqrt(mean_squared_error(y_train, lgb_oof_preds))
print(f'\nLightGBM OOF RMSE: {lgb_oof_rmse:.4f}')
print(f'Fold RMSE: {[f"{r:.4f}" for r in lgb_fold_rmse]}')

## 6. Model B: XGBoost

In [ ]:
xgb_params = {
    'objective':        'reg:squarederror',
    'eval_metric':      'rmse',
    'learning_rate':    0.03,
    'max_depth':        7,
    'min_child_weight': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'n_estimators':     3000,
    'early_stopping_rounds': 50,
    'random_state':     SEED,
    'n_jobs':           -1,
    'device':           'cpu',
    'verbosity':        0,
}

xgb_oof_preds  = np.zeros(len(X_train))
xgb_test_preds = np.zeros(len(X_test))
xgb_fold_rmse  = []

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)
    
    val_pred = model.predict(X_val)
    xgb_oof_preds[val_idx] = val_pred
    xgb_test_preds += model.predict(X_test) / N_FOLDS
    
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    xgb_fold_rmse.append(rmse)
    print(f'  Fold {fold_i} XGB RMSE: {rmse:.4f}')

xgb_oof_rmse = np.sqrt(mean_squared_error(y_train, xgb_oof_preds))
print(f'\nXGBoost OOF RMSE: {xgb_oof_rmse:.4f}')

## 7. Model C: CatBoost

In [ ]:
cat_oof_preds  = np.zeros(len(X_train))
cat_test_preds = np.zeros(len(X_test))
cat_fold_rmse  = []

for fold_i, (tr_idx, val_idx) in enumerate(splits):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3,
        loss_function='RMSE',
        eval_metric='RMSE',
        early_stopping_rounds=50,
        random_seed=SEED,
        verbose=200,
    )
    
    model.fit(
        Pool(X_tr, y_tr),
        eval_set=Pool(X_val, y_val),
    )
    
    val_pred = model.predict(X_val)
    cat_oof_preds[val_idx] = val_pred
    cat_test_preds += model.predict(X_test) / N_FOLDS
    
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    cat_fold_rmse.append(rmse)
    print(f'  Fold {fold_i} CAT RMSE: {rmse:.4f}')

cat_oof_rmse = np.sqrt(mean_squared_error(y_train, cat_oof_preds))
print(f'\nCatBoost OOF RMSE: {cat_oof_rmse:.4f}')

## 8. Ensemble: Weighted Blend

Find optimal weights by minimizing OOF RMSE.

In [ ]:
from scipy.optimize import minimize

oof_matrix = np.column_stack([lgb_oof_preds, xgb_oof_preds, cat_oof_preds])
test_matrix = np.column_stack([lgb_test_preds, xgb_test_preds, cat_test_preds])
y_true = y_train.values

def ensemble_rmse(weights):
    w = np.array(weights)
    w = w / w.sum()  # normalize
    blend = oof_matrix @ w
    return np.sqrt(mean_squared_error(y_true, blend))

res = minimize(
    ensemble_rmse,
    x0=[1/3, 1/3, 1/3],
    method='Nelder-Mead',
    options={'maxiter': 1000, 'xatol': 1e-6, 'fatol': 1e-6}
)

best_weights = res.x / res.x.sum()
print(f'Optimal weights: LGB={best_weights[0]:.3f}  XGB={best_weights[1]:.3f}  CAT={best_weights[2]:.3f}')

ensemble_oof   = oof_matrix  @ best_weights
ensemble_test  = test_matrix @ best_weights
ensemble_rmse_val = np.sqrt(mean_squared_error(y_true, ensemble_oof))

print(f'\nModel comparison (OOF RMSE):')
print(f'  LightGBM  : {lgb_oof_rmse:.4f}')
print(f'  XGBoost   : {xgb_oof_rmse:.4f}')
print(f'  CatBoost  : {cat_oof_rmse:.4f}')
print(f'  Ensemble  : {ensemble_rmse_val:.4f}  ← final model')

## 9. Feature Importance

In [ ]:
# Use the last LGB model from CV for feature importance
# (re-run only last fold for speed; for production, average importances across folds)
tr_idx, val_idx = splits[-1]
dtrain_full = lgb.Dataset(X_train, label=y_train)
lgb_final = lgb.train(
    {**lgb_params, 'learning_rate': 0.01},
    dtrain_full,
    num_boost_round=int(model.best_iteration * 1.1) if hasattr(model, 'best_iteration') else 1000,
    valid_sets=[dtrain_full],
    callbacks=[lgb.log_evaluation(period=500)]
)

imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_final.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
top_n = 30
imp.head(top_n).plot.barh(x='feature', y='importance', ax=ax, legend=False,
                           color='steelblue')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Feature Importances (LightGBM, gain)')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.show()

print(imp.head(20).to_string(index=False))

## 10. OOF Prediction Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs actual
sample_idx = np.random.choice(len(y_true), size=min(5000, len(y_true)), replace=False)
axes[0].scatter(y_true[sample_idx], ensemble_oof[sample_idx], alpha=0.3, s=4)
lo, hi = y_true.min(), y_true.max()
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=2)
axes[0].set_title('Predicted vs Actual TVT (ensemble OOF)')
axes[0].set_xlabel('Actual TVT')
axes[0].set_ylabel('Predicted TVT')

# Residuals
residuals = y_true - ensemble_oof
axes[1].hist(residuals, bins=100, color='steelblue', edgecolor='none')
axes[1].axvline(0, color='red', lw=2)
axes[1].set_title(f'Residuals (mean={residuals.mean():.3f}, std={residuals.std():.3f})')
axes[1].set_xlabel('Residual (m)')

# Residuals vs predicted
axes[2].scatter(ensemble_oof[sample_idx], residuals[sample_idx], alpha=0.3, s=4)
axes[2].axhline(0, color='red', lw=1.5)
axes[2].set_title('Residuals vs Predicted')
axes[2].set_xlabel('Predicted TVT')
axes[2].set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 11. Per-Well CV Analysis

In [ ]:
train_with_preds = train_fe[[WELL_COL, TARGET_COL]].copy()
train_with_preds['pred'] = ensemble_oof
train_with_preds['sq_err'] = (train_with_preds[TARGET_COL] - train_with_preds['pred'])**2

well_rmse = train_with_preds.groupby(WELL_COL)['sq_err'].mean().apply(np.sqrt).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(well_rmse, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Per-Well RMSE Distribution')
axes[0].set_xlabel('RMSE (m)')
axes[0].axvline(well_rmse.median(), color='red', ls='--', label=f'Median: {well_rmse.median():.2f}')
axes[0].legend()

# Top 15 worst wells
worst_wells = well_rmse.head(15)
axes[1].barh(worst_wells.index.astype(str), worst_wells.values, color='tomato')
axes[1].invert_yaxis()
axes[1].set_title('Top 15 Worst Wells (by RMSE)')
axes[1].set_xlabel('RMSE (m)')

plt.tight_layout()
plt.show()

print(f'Median per-well RMSE : {well_rmse.median():.4f}')
print(f'P90 per-well RMSE    : {well_rmse.quantile(0.9):.4f}')

## 12. Generate Submission

In [ ]:
# Inspect sample submission format
print(sample_sub.head())
print(f'\nSample submission columns: {list(sample_sub.columns)}')

In [ ]:
# Build submission — adjust column names to match sample_submission.csv
id_col = sample_sub.columns[0]    # e.g. 'row_id' or 'id'
pred_col = sample_sub.columns[1]  # e.g. 'TVT'

submission = pd.DataFrame({
    id_col:   test_fe[id_col].values if id_col in test_fe.columns else sample_sub[id_col].values,
    pred_col: ensemble_test
})

# Sanity check: same shape as sample submission
assert len(submission) == len(sample_sub), \
    f'Submission length mismatch: {len(submission)} vs {len(sample_sub)}'

submission_path = SUBMIT_DIR / 'submission_ensemble.csv'
submission.to_csv(submission_path, index=False)

print(f'Submission saved: {submission_path}')
print(f'Shape: {submission.shape}')
print(submission.head(10))
print(f'\nPrediction stats:')
print(submission[pred_col].describe().round(3))

## 13. DWT Feature Analysis

Visualize what the Discrete Wavelet Transform is capturing in the GR signal.

In [ ]:
# Pick one well and visualize DWT decomposition
sample_well = train[WELL_COL].unique()[0]
well_df = train_fe[train_fe[WELL_COL] == sample_well].sort_values(DEPTH_COL).head(500)

dwt_cols = [c for c in well_df.columns if c.startswith('dwt_')]

if dwt_cols:
    fig, axes = plt.subplots(len(dwt_cols) + 1, 1, figsize=(14, 3 * (len(dwt_cols) + 1)), sharex=True)
    
    axes[0].plot(well_df[DEPTH_COL], well_df['GR_norm'], color='darkorange', lw=0.8)
    axes[0].set_title(f'Well {sample_well} — Normalized GR (original signal)')
    axes[0].set_ylabel('GR norm')
    
    for i, col in enumerate(dwt_cols, 1):
        axes[i].plot(well_df[DEPTH_COL], well_df[col], lw=0.8)
        axes[i].set_title(f'DWT component: {col}')
        axes[i].set_ylabel('Amplitude')
    
    axes[-1].set_xlabel('Measured Depth (m)')
    plt.tight_layout()
    plt.show()
else:
    print('DWT features not available (PyWavelets not installed or too few samples).')

## 14. Advanced: LSTM Sequence Model (Optional)

The GR-to-TVT relationship is a sequence problem. An LSTM can capture long-range dependencies that tree models miss.

In [ ]:
# Uncomment to run LSTM training (requires tensorflow or torch)

# import tensorflow as tf
# from tensorflow import keras
#
# SEQ_LEN = 50  # context window in depth samples
#
# def make_sequences(df, well_col, depth_col, feature_cols, target_col, seq_len):
#     X_seqs, y_seqs, well_ids = [], [], []
#     for well_id, gdf in df.groupby(well_col):
#         gdf = gdf.sort_values(depth_col)
#         feat = gdf[feature_cols].values
#         tgt  = gdf[target_col].values
#         for i in range(seq_len, len(gdf)):
#             X_seqs.append(feat[i-seq_len:i])
#             y_seqs.append(tgt[i])
#             well_ids.append(well_id)
#     return np.array(X_seqs), np.array(y_seqs), np.array(well_ids)
#
# X_seq, y_seq, well_seq = make_sequences(
#     train_fe, WELL_COL, DEPTH_COL,
#     [c for c in feature_cols if 'lag' not in c],  # avoid double lags
#     TARGET_COL, SEQ_LEN
# )
#
# model = keras.Sequential([
#     keras.layers.LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, X_seq.shape[2])),
#     keras.layers.LSTM(64),
#     keras.layers.Dense(32, activation='relu'),
#     keras.layers.Dense(1)
# ])
# model.compile(optimizer='adam', loss='mse')
# model.summary()

print('LSTM code block is commented out — uncomment and install TensorFlow to run.')

## 15. Summary & Next Steps

### What we built

| Step | Details |
|---|---|
| Feature engineering | Rolling stats (5 windows), lag features (7 lags), DWT (4 components), 3D tortuosity, GR normalization |
| Validation | 5-fold GroupKFold by well |
| Models | LightGBM + XGBoost + CatBoost |
| Ensemble | Optimized weighted blend via Nelder-Mead |

### Ideas to improve further

1. **Tune DWT wavelet basis** (try `sym4`, `coif3`) and level (3-5)
2. **Add reference well features**: correlate horizontal GR pattern with vertical type well
3. **Particle filter post-processing**: enforce TVT trajectory smoothness (physics constraint)
4. **LSTM/Transformer**: train on full-sequence well data for better temporal modeling
5. **Stacking**: use OOF predictions of base models as input to a meta-learner
6. **Hyperparameter optimization**: Optuna on LGBM/XGB with 50 trials
7. **Pseudo-labeling**: use confident test predictions to augment training data

### Expected LB improvement path

```
LightGBM basic          → ~12–15
+ Feature engineering   → ~10–12
+ DWT features          → ~9–10
+ Ensemble              → ~8.5–9.5
+ Reference well corr.  → ~8–9
+ Particle filter       → <8 (top tier)
```